# CAFE paper — build every table and figure

Run this notebook top to bottom. It regenerates all 10 tables and all 5 figures from the
recorded result files, then copies them into **`Final/paper_assets/`** named to match the
captions already in the Word document:

| File it writes | Goes in the slot marked |
|---|---|
| `Table_01.png` … `Table_10.png` | `[ insert Table_01.png here ]` … |
| `Figure_1.png` … `Figure_5.png` | `[ insert Figure_1.png here ]` … |

**In Word:** click the grey `[ insert … ]` line, delete it, then *Insert → Pictures* and
pick the matching file. The caption above (tables) or below (figures) is already written.

Nothing here fits a model. Every number is read from the files the experiments wrote, so
re-running this cannot change a result — it can only redraw it.

## 1. Setup

Use the project environment (`.venv`). If the kernel is the system Python this will fail
on `import pandas` — switch kernels and re-run.

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

HERE = Path.cwd()
REPO = HERE if (HERE / "synthetic_generator_tuned.py").exists() else HERE.parent
assert (REPO / "synthetic_generator_tuned.py").exists(), (
    f"Run this notebook from the repo root or from Final/. Looked in: {REPO}"
)

STUDY = REPO / "results" / "xmodal_study_2026-07-22"
FIGS = REPO / "figures" / "cafe"
TABLES = FIGS / "tables"
OUT = REPO / "Final" / "paper_assets"
OUT.mkdir(parents=True, exist_ok=True)

PY = REPO / ".venv" / "Scripts" / "python.exe"
if not PY.exists():
    PY = Path(sys.executable)

print(f"repo    {REPO}")
print(f"python  {PY}")
print(f"output  {OUT}")

## 2. Regenerate the figures and tables

Both scripts read the recorded per-fold CSVs and redraw from scratch, so this is safe to
run as often as you like.

In [ ]:
for script in ("make_cafe_figures.py", "make_cafe_tables.py"):
    print(f"=== {script} " + "=" * (52 - len(script)))
    r = subprocess.run([str(PY), str(STUDY / script)], cwd=REPO,
                       capture_output=True, text=True)
    print(r.stdout.strip() or "(no output)")
    if r.returncode != 0:
        print(r.stderr.strip())
        raise SystemExit(f"{script} failed")
print("\nall assets rebuilt")

## 3. Copy them out under their paper numbers

The scripts write descriptive names; the paper refers to numbers. This maps one to the
other so you never have to guess which picture is Table 6.

In [ ]:
# (paper label, source file, what it shows)
ASSETS = [
    ("Table_01", TABLES / "table01_feature_families.png",
     "The five feature families"),
    ("Table_02", TABLES / "table02_encoder_specs.png",
     "Encoder specifications"),
    ("Table_03", TABLES / "table03_discrimination_calibration.png",
     "Discrimination, overall error and calibration"),
    ("Table_04", TABLES / "table04_engineered_features_by_learner.png",
     "Engineered features, by learner"),
    ("Table_05", TABLES / "table05_encoder_b_ablation.png",
     "Encoder B ablation"),
    ("Table_06", TABLES / "table06_decision_curve.png",
     "Net benefit at representative thresholds"),
    ("Table_07", TABLES / "table07_operating_point.png",
     "90 %-sensitivity screening threshold"),
    ("Table_08", TABLES / "table08_shap_top15.png",
     "Top 15 features by mean |SHAP|"),
    ("Table_09", TABLES / "table09_cost.png",
     "Training and inference cost"),
    ("Table_10", TABLES / "table10_ablation_decomposition.png",
     "Decomposition of the final AUROC"),
    ("Figure_1", FIGS / "fig_cafe_calibration.png",
     "Reliability curves"),
    ("Figure_2", FIGS / "fig_cafe_decision_curve.png",
     "Net benefit against risk threshold"),
    ("Figure_3", FIGS / "fig_cafe_shap.png",
     "Top 20 features by mean |SHAP|"),
    ("Figure_4", FIGS / "fig_cafe_accuracy_vs_cost.png",
     "AUROC against training time"),
    ("Figure_5", FIGS / "fig_cafe_ablation.png",
     "The decomposition as a waterfall"),
]

missing = [str(src) for _, src, _ in ASSETS if not src.exists()]
assert not missing, "these did not get built:\n  " + "\n  ".join(missing)

for label, src, what in ASSETS:
    dst = OUT / f"{label}.png"
    shutil.copyfile(src, dst)
    print(f"  {label}.png   {what}")

print(f"\n{len(ASSETS)} files written to {OUT}")

## 4. Look at them

Check each one before you paste it in — this is the last point where a wrong picture is
cheap to catch.

In [ ]:
from IPython.display import Image, Markdown, display

for label, _, what in ASSETS:
    pretty = label.replace("_0", " ").replace("_", " ")
    display(Markdown(f"### {pretty} — {what}"))
    display(Image(filename=str(OUT / f"{label}.png"), width=780))

## 5. Confirm the manuscript's numbers still match their sources

Optional, but it is the check that catches a figure drifting away from the result it is
supposed to show. It must print **ALL NUMBERS … MATCH**.

In [ ]:
r = subprocess.run([str(PY), str(STUDY / "verify_cafe_manuscript.py")], cwd=REPO,
                   capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip())